# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All entities are referenced by their `@id` per Croissant best practices.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
print(f"Dataset Title: {ds.metadata.name}\n\nDescription: {ds.metadata.description}")

## 2. Data Overview
Review available record sets, their IDs, and associated fields.

In [ ]:
# Explore available record sets by their `@id`
print('Record Sets available:')
for rs in ds.metadata.record_sets:
    print(f"  @id: {rs.id} | name: {rs.name}")

# For demonstration, inspect the first record set's fields/columns
first_rs = ds.metadata.record_sets[0]
print(f"\nFields for record set '@id': {first_rs.id} - {first_rs.name}")
for field in first_rs.fields:
    col_info = f"column: {field.column.id}" if field.column is not None else "(no column)"
    print(f"  @id: {field.id} | name: {field.name} | {col_info} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references by `@id`.

In [ ]:
# Extract data from each record set into a DataFrame
record_set_ids = [rs.id for rs in ds.metadata.record_sets]
dfs = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set {rs_id} ...")
    records = list(ds.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"  Columns: {list(df.columns)} | Records: {len(df)}")

# Preview the columns and first rows for the main record set
main_rs_id = record_set_ids[0]
print(f"\nAvailable columns in main record set (@id: {main_rs_id}):\n{dfs[main_rs_id].columns.tolist()}")
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping, referencing all fields by their respective `@id`.

In [ ]:
# Pick a numeric field (by @id or name from overview cell above)
numeric_field_id = None
for field in ds.metadata.record_sets[0].fields:
    # Pick the first Float/Integer/Number field as an example
    if field.data_type in ["schema:Float", "schema:Integer", "schema:Number"]:
        numeric_field_id = field.id
        numeric_field_col = field.column.id if field.column is not None else field.name
        break

# Show which field was chosen
print(f"Numeric field chosen for analysis: {numeric_field_id} (column: {numeric_field_col})")

df = dfs[main_rs_id]

# Filter: remove rows with missing or invalid values
if numeric_field_col in df.columns:
    # Example: Use 10th percentile as threshold if enough values exist, else 0
    try:
        threshold = df[numeric_field_col].dropna().quantile(0.10)
    except Exception:
        threshold = 0
    filtered_df = df[df[numeric_field_col] > threshold].copy()
    print(f"Filtered records with {numeric_field_col} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df[[numeric_field_col]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_col}_normalized"] = (
        filtered_df[numeric_field_col] - filtered_df[numeric_field_col].mean()
    ) / filtered_df[numeric_field_col].std()
    print(f"\nNormalized {numeric_field_col} for filtered records:")
    print(filtered_df[[numeric_field_col, f"{numeric_field_col}_normalized"]].head())

    # Attempt to group by a categorical field
    group_field_id = None
    for field in ds.metadata.record_sets[0].fields:
        if field.data_type in ["schema:Text", "@vocab"] and field.column is not None:
            group_field_id = field.id
            group_field_col = field.column.id
            break
    if group_field_id and group_field_col in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_col)[numeric_field_col].mean().reset_index()
        )
        print(f"\nGrouped mean of {numeric_field_col} by {group_field_col}:")
        print(grouped_df.head())
else:
    print("No numeric column found for EDA.")

## 5. Visualization
Visualize data distributions or relationships for any numeric field extracted above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_col in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_col].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_col}")
    plt.xlabel(numeric_field_col)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if 'group_field_col' in locals() and group_field_col in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_col, y=numeric_field_col)
        plt.title(f"{numeric_field_col} by {group_field_col}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using its Croissant schema, explored and inspected the available record sets by their `@id`, extracted tabular data for analysis, and performed basic EDA including filtering, normalization, grouping, and visualization on a selected numeric field. Further in-depth domain-specific analyses can be performed leveraging the field metadata provided by Croissant.

*Tip: Use the record set and field `@id` consistently for robust, reproducible processing scripts across FAIR datasets!*